# Project 002: The Classification Engine
## Notebook 1: Ancient Manuscript Authenticity Classification

**The AI Engineering Lab** | Post 2 of the Progressive AIML Series

---

### Problem Statement

Given physical and chemical measurements of an ancient manuscript, predict whether it is **authentic** (1) or **forged** (0). This is a binary classification problem with significant class imbalance — most manuscripts in a dataset are forged (80%), which makes naive accuracy misleading and requires careful metric selection.

### What This Notebook Covers

1. Exploratory Data Analysis (EDA) with class balance analysis
2. Preprocessing pipeline: StandardScaler + OneHotEncoder
3. Logistic Regression from scratch: Sigmoid + Cross-Entropy + Gradient Descent
4. sklearn LogisticRegression with L1/L2 regularization
5. Handling class imbalance: class_weight and SMOTE
6. Full metrics suite: Accuracy, Precision, Recall, F1, AUC-ROC, AUC-PR, MCC
7. Threshold analysis: finding the optimal classification threshold
8. Probability calibration: Platt Scaling

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = {'authentic': '#4CAF50', 'forged': '#F44336', 'primary': '#2196F3', 'secondary': '#FF9800'}

print('Libraries loaded successfully.')

## 1. Load and Explore the Dataset

We begin with a thorough EDA to understand the data distribution, feature relationships, and the degree of class imbalance. Understanding class imbalance upfront is critical — it determines whether accuracy is a valid metric and whether we need resampling strategies.

In [ ]:
df = pd.read_csv('../data/manuscript_authenticity_data.csv')
print('Shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())
print('\nData types and null counts:')
display(df.info())

In [ ]:
print('Descriptive Statistics:')
display(df.describe().round(3))

In [ ]:
# Class balance analysis
vc = df['is_authentic'].value_counts()
print(f'Class Distribution:')
print(f'  Authentic (1): {vc[1]} ({vc[1]/len(df)*100:.1f}%)')
print(f'  Forged    (0): {vc[0]} ({vc[0]/len(df)*100:.1f}%)')
print(f'  Imbalance ratio: {vc[0]/vc[1]:.1f}:1')
print()
print('IMPORTANT: With 80% forged, a naive classifier that predicts forged for everything')
print('would achieve 80% accuracy. This is why accuracy alone is misleading for imbalanced data.')

In [ ]:
# Feature distributions by class
features = ['ink_iron_ratio', 'parchment_density', 'carbon_14_ratio',
            'scribal_pressure_var', 'uv_fluorescence_idx', 'linguistic_anachronism_score']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    authentic = df[df['is_authentic'] == 1][feat]
    forged    = df[df['is_authentic'] == 0][feat]
    axes[i].hist(authentic, bins=40, alpha=0.6, color=COLORS['authentic'], label='Authentic', density=True)
    axes[i].hist(forged,    bins=40, alpha=0.6, color=COLORS['forged'],    label='Forged',   density=True)
    axes[i].set_title(feat.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Feature Distributions by Class: Authentic vs Forged Manuscripts',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. The Sigmoid Function: From Linear Scores to Probabilities

Logistic Regression does not directly predict a class. Instead, it first computes a linear score (the log-odds or logit), then maps it through the sigmoid function to produce a probability between 0 and 1. The sigmoid function is: σ(z) = 1 / (1 + e^(-z)). When z is large and positive, σ(z) → 1 (predict class 1). When z is large and negative, σ(z) → 0 (predict class 0). The decision boundary is at z = 0, where σ(z) = 0.5.

In [ ]:
def sigmoid(z):
    """The sigmoid (logistic) function: maps any real number to (0, 1)."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

# Visualize the sigmoid function
z = np.linspace(-8, 8, 300)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(z, sigmoid(z), color=COLORS['primary'], linewidth=3)
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Decision boundary (σ=0.5)')
axes[0].axvline(0.0, color='gray', linestyle='--', linewidth=1.0)
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z > 0), alpha=0.15, color=COLORS['authentic'], label='Predict Authentic')
axes[0].fill_between(z, sigmoid(z), 0.5, where=(z < 0), alpha=0.15, color=COLORS['forged'],    label='Predict Forged')
axes[0].set_xlabel('Log-odds (z = wᵀx + b)', fontsize=11)
axes[0].set_ylabel('Predicted Probability P(y=1|x)', fontsize=11)
axes[0].set_title('The Sigmoid Function', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Derivative of sigmoid
dsigmoid = sigmoid(z) * (1 - sigmoid(z))
axes[1].plot(z, dsigmoid, color=COLORS['secondary'], linewidth=3)
axes[1].axvline(0, color='gray', linestyle='--', linewidth=1.0, label='Max gradient at z=0')
axes[1].set_xlabel('z', fontsize=11)
axes[1].set_ylabel("σ'(z) = σ(z)(1 - σ(z))", fontsize=11)
axes[1].set_title('Sigmoid Derivative (used in Backpropagation)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sigmoid Function and Its Derivative', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Cross-Entropy Loss: Why Not MSE for Classification?

For regression, we minimized MSE. For classification, we use **Binary Cross-Entropy (Log Loss)**. The reason: MSE combined with sigmoid creates a non-convex loss surface with many local minima, making gradient descent unreliable. Cross-entropy is convex and has a single global minimum. Mathematically: L = -[y·log(ŷ) + (1-y)·log(1-ŷ)]. When y=1 and ŷ→0, the loss → ∞ (heavily penalizes confident wrong predictions). When y=1 and ŷ→1, the loss → 0.

In [ ]:
def cross_entropy_loss(y_true, y_pred_prob, eps=1e-15):
    """Binary cross-entropy loss."""
    y_pred_prob = np.clip(y_pred_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred_prob) + (1 - y_true) * np.log(1 - y_pred_prob))

# Visualize cross-entropy vs MSE for classification
p = np.linspace(0.001, 0.999, 300)
ce_y1 = -np.log(p)           # Cross-entropy when true label = 1
ce_y0 = -np.log(1 - p)       # Cross-entropy when true label = 0
mse_y1 = (1 - p)**2
mse_y0 = p**2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(p, ce_y1, color=COLORS['authentic'], linewidth=2.5, label='CE: y=1 (correct → 0 loss)')
axes[0].plot(p, ce_y0, color=COLORS['forged'],    linewidth=2.5, label='CE: y=0 (correct → 0 loss)')
axes[0].set_title('Cross-Entropy Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Probability ŷ')
axes[0].set_ylabel('Loss')
axes[0].set_ylim(0, 5)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(p, mse_y1, color=COLORS['authentic'], linewidth=2.5, label='MSE: y=1')
axes[1].plot(p, mse_y0, color=COLORS['forged'],    linewidth=2.5, label='MSE: y=0')
axes[1].set_title('MSE Loss (not ideal for classification)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Probability ŷ')
axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Cross-Entropy vs MSE: Why CE is Better for Classification', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Cross-entropy penalizes confident wrong predictions much more heavily than MSE.')
print('This creates stronger gradient signals and faster, more reliable learning.')

## 4. Logistic Regression from Scratch

Before using sklearn, we implement logistic regression manually. The gradient update rule for cross-entropy loss is elegantly simple: ∂L/∂w = (1/n) · Xᵀ · (ŷ - y). This is the same form as linear regression gradient descent — the difference is that ŷ = σ(Xw + b) instead of ŷ = Xw + b.

In [ ]:
class LogisticRegressionScratch:
    """
    Logistic Regression implemented from scratch using gradient descent.
    Demonstrates the core math: sigmoid + cross-entropy + gradient update.
    """
    def __init__(self, lr=0.1, n_iter=1000, lambda_reg=0.01):
        self.lr = lr
        self.n_iter = n_iter
        self.lambda_reg = lambda_reg  # L2 regularization strength
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iter):
            # Forward pass: compute predictions
            z = X @ self.weights + self.bias
            y_hat = self._sigmoid(z)

            # Compute cross-entropy loss with L2 regularization
            eps = 1e-15
            loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
            loss += (self.lambda_reg / (2 * n_samples)) * np.sum(self.weights**2)
            self.loss_history.append(loss)

            # Backward pass: compute gradients
            error = y_hat - y
            dw = (X.T @ error) / n_samples + (self.lambda_reg / n_samples) * self.weights
            db = np.mean(error)

            # Update parameters
            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

print('LogisticRegressionScratch class defined.')

## 5. Preprocessing and Train/Test Split

We use `stratify=y` in train_test_split to ensure the class distribution is preserved in both train and test sets. This is critical for imbalanced datasets — without stratification, the test set might accidentally contain very few authentic samples, making evaluation unreliable.

In [ ]:
FEATURES = ['ink_iron_ratio', 'parchment_density', 'carbon_14_ratio',
            'scribal_pressure_var', 'pigment_layer_count', 'uv_fluorescence_idx',
            'linguistic_anachronism_score', 'vellum_thickness_mm']
TARGET = 'is_authentic'

X = df[FEATURES].values
y = df[TARGET].values

# Stratified split: preserves class ratio in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler: zero mean, unit variance
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {X_train_s.shape}, Test: {X_test_s.shape}')
print(f'Train class ratio: {y_train.mean():.3f} authentic')
print(f'Test  class ratio: {y_test.mean():.3f} authentic (stratification preserved)')

In [ ]:
# Train from-scratch model
lr_scratch = LogisticRegressionScratch(lr=0.5, n_iter=2000, lambda_reg=0.1)
lr_scratch.fit(X_train_s, y_train)

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lr_scratch.loss_history, color=COLORS['primary'], linewidth=2)
ax.set_title('Gradient Descent Convergence: Cross-Entropy Loss', fontsize=12, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

y_pred_scratch = lr_scratch.predict(X_test_s)
print(f'From-scratch model accuracy: {accuracy_score(y_test, y_pred_scratch):.4f}')

## 6. Handling Class Imbalance: SMOTE

With 80% forged manuscripts, the model learns to predict 'forged' for almost everything and still achieves 80% accuracy. SMOTE (Synthetic Minority Oversampling Technique) generates synthetic samples of the minority class (authentic) by interpolating between existing minority samples in feature space. This balances the training set without simply duplicating existing samples.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)

print(f'Before SMOTE: {X_train_s.shape[0]} samples, {y_train.mean():.3f} authentic ratio')
print(f'After  SMOTE: {X_train_sm.shape[0]} samples, {y_train_sm.mean():.3f} authentic ratio')

## 7. sklearn Logistic Regression with Regularization

sklearn's LogisticRegression uses the `C` parameter (inverse of regularization strength: C = 1/λ). A small C means strong regularization (simpler model), a large C means weak regularization (complex model). We compare L1 (Lasso-like, can zero out features) and L2 (Ridge-like, shrinks all features) penalties.

In [ ]:
results = {}

# Baseline: no resampling, default C=1, L2
lr_base = LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', max_iter=1000, random_state=42)
lr_base.fit(X_train_s, y_train)
results['Baseline (L2, C=1)'] = lr_base

# With class_weight='balanced'
lr_balanced = LogisticRegression(C=1.0, penalty='l2', class_weight='balanced',
                                  solver='lbfgs', max_iter=1000, random_state=42)
lr_balanced.fit(X_train_s, y_train)
results['Balanced Weights (L2)'] = lr_balanced

# L1 with SMOTE
lr_l1 = LogisticRegression(C=0.5, penalty='l1', solver='liblinear', max_iter=1000, random_state=42)
lr_l1.fit(X_train_sm, y_train_sm)
results['L1 + SMOTE (C=0.5)'] = lr_l1

# L2 with SMOTE + CV-tuned C
lr_cv = LogisticRegressionCV(Cs=20, cv=5, penalty='l2', scoring='f1',
                              solver='lbfgs', max_iter=1000, random_state=42)
lr_cv.fit(X_train_sm, y_train_sm)
results[f'L2 + SMOTE + CV (C={lr_cv.C_[0]:.3f})'] = lr_cv

print('Models trained. Evaluating...')
print()

# Evaluation table
eval_rows = []
for name, model in results.items():
    X_te = X_test_s
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    eval_rows.append({
        'Model': name,
        'Accuracy': f'{accuracy_score(y_test, y_pred):.4f}',
        'Precision': f'{precision_score(y_test, y_pred):.4f}',
        'Recall': f'{recall_score(y_test, y_pred):.4f}',
        'F1': f'{f1_score(y_test, y_pred):.4f}',
        'AUC-ROC': f'{roc_auc_score(y_test, y_prob):.4f}',
        'MCC': f'{matthews_corrcoef(y_test, y_pred):.4f}',
    })

display(pd.DataFrame(eval_rows))

## 8. Threshold Analysis

The default classification threshold is 0.5, but this is rarely optimal for imbalanced datasets. By varying the threshold, we can trade off between precision and recall depending on the application. For manuscript authentication, missing a forgery (false negative) might be more costly than a false alarm, so we may prefer a lower threshold to maximize recall.

In [ ]:
best_model = lr_cv
y_prob_best = best_model.predict_proba(X_test_s)[:, 1]

thresholds = np.linspace(0.05, 0.95, 100)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, precisions, color=COLORS['primary'],    linewidth=2, label='Precision')
ax.plot(thresholds, recalls,    color=COLORS['authentic'],  linewidth=2, label='Recall')
ax.plot(thresholds, f1s,        color=COLORS['secondary'],  linewidth=2.5, label='F1 Score')
ax.axvline(best_t, color='red', linestyle='--', linewidth=2, label=f'Optimal threshold = {best_t:.2f}')
ax.axvline(0.5,    color='gray', linestyle=':', linewidth=1.5, label='Default threshold = 0.50')
ax.set_xlabel('Classification Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Analysis: Precision, Recall, and F1 vs Threshold', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Optimal threshold (max F1): {best_t:.2f}')
y_pred_opt = (y_prob_best >= best_t).astype(int)
print(f'F1 at optimal threshold: {f1_score(y_test, y_pred_opt):.4f}')
print(f'F1 at default threshold: {f1_score(y_test, (y_prob_best >= 0.5).astype(int)):.4f}')

## 9. ROC and Precision-Recall Curves

The ROC curve plots True Positive Rate (Recall) vs False Positive Rate at all thresholds. AUC-ROC = 1.0 is perfect; 0.5 is random. The Precision-Recall (PR) curve is more informative for imbalanced datasets because it focuses on the minority class performance. AUC-PR = 1.0 is perfect; the baseline is the class prevalence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_colors = [COLORS['forged'], COLORS['secondary'], COLORS['authentic'], COLORS['primary']]
for (name, model), color in zip(results.items(), model_colors):
    y_prob = model.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0,1],[0,1], 'k--', linewidth=1, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[0].set_title('ROC Curves', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.3)

for (name, model), color in zip(results.items(), model_colors):
    y_prob = model.predict_proba(X_test_s)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f'{name} (AP={ap:.3f})')

baseline = y_test.mean()
axes[1].axhline(baseline, color='gray', linestyle='--', linewidth=1.5, label=f'Baseline (AP={baseline:.3f})')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('ROC and Precision-Recall Curves: All Models Compared', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Confusion Matrix and Final Metrics

The confusion matrix breaks down predictions into four categories: True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN). From these, all classification metrics are derived. Matthews Correlation Coefficient (MCC) is the most balanced metric for imbalanced datasets — it considers all four quadrants of the confusion matrix.

In [ ]:
y_pred_final = (y_prob_best >= best_t).astype(int)
cm = confusion_matrix(y_test, y_pred_final)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Forged', 'Authentic'], yticklabels=['Forged', 'Authentic'])
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized confusion matrix
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Greens', ax=axes[1],
            xticklabels=['Forged', 'Authentic'], yticklabels=['Forged', 'Authentic'])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle(f'Confusion Matrix at Optimal Threshold = {best_t:.2f}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nFull Classification Report:')
print(classification_report(y_test, y_pred_final, target_names=['Forged', 'Authentic']))
print(f'Matthews Correlation Coefficient: {matthews_corrcoef(y_test, y_pred_final):.4f}')
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'Average Precision (AUC-PR): {average_precision_score(y_test, y_prob_best):.4f}')

## 11. Feature Importance via Coefficients

In logistic regression, the model coefficients directly represent the log-odds contribution of each feature. A positive coefficient means the feature increases the probability of class 1 (authentic). A negative coefficient means it decreases it. After standardization, the magnitude of coefficients is directly comparable across features.

In [ ]:
coefs = lr_cv.coef_[0]
feat_importance = pd.DataFrame({'Feature': FEATURES, 'Coefficient': coefs})
feat_importance = feat_importance.sort_values('Coefficient')

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = [COLORS['authentic'] if c > 0 else COLORS['forged'] for c in feat_importance['Coefficient']]
ax.barh(feat_importance['Feature'], feat_importance['Coefficient'], color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Logistic Regression Coefficients (Standardized Features)\nPositive = Increases P(Authentic)', fontsize=12, fontweight='bold')
ax.set_xlabel('Coefficient Value (Log-Odds Contribution)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the complete logistic regression pipeline on the manuscript authenticity dataset:

| Concept | Key Takeaway |
|:---|:---|
| Sigmoid function | Maps log-odds to probabilities; decision boundary at σ(z) = 0.5 |
| Cross-entropy loss | Convex, penalizes confident wrong predictions; better than MSE for classification |
| Class imbalance | Accuracy is misleading; use F1, AUC-PR, MCC; SMOTE and class_weight help |
| Threshold tuning | Default 0.5 is rarely optimal; tune based on business cost of FP vs FN |
| L1 vs L2 | L1 can zero out features; L2 shrinks all; C = 1/λ controls strength |
| MCC | Most balanced single metric for imbalanced binary classification |

**Next:** Notebook 02 applies the same pipeline to silicon timing test pass/fail prediction with a different class imbalance pattern and a post-silicon validation context.